# Regresi teks + tabel: OOF Ridge + kelas-bin + tetangga → LightGBM
**Kapan dipakai:** target **angka** (harga, skor, jumlah, rating sebagai angka) dari tabel yang
punya kolom teks dan kolom angka/kategori. Pasangan regresi dari `stacking_teks_tabular`.

1. **Ridge OOF** di TF-IDF kata + huruf: tebakan angka dari teks (pola "ridge OOF" juara lomba
   teks+tabel Kaggle, mis. Mercari Price, Avito Demand).
2. **Kelas-bin OOF**: target dipotong jadi `N_BINS` kuantil, LogReg menebak peluang tiap bin.
   Menangkap bentuk sebaran (mis. "ulasan ini antara 1 atau 5") yang hilang di satu angka.
3. **Tetangga terdekat**: rata-rata target tetangga (berbobot kemiripan), sebarannya, target
   tetangga ke-1, kemiripan tertinggi (kembar → lookup).

Semua dihitung di dalam lipatan `KFold(5, shuffle, 42)` (sama dengan `finetune_regresi`), lalu
LightGBM memakai fitur itu + kolom tabel.
**Data demo:** ulasan baju, target `rating` sebagai angka.

In [ ]:
import copy
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from scipy.sparse import hstack
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors

## Konfigurasi
`LOG_TARGET = True` untuk target miring ke kanan (harga, jumlah): model belajar `log1p(y)`,
tebakan dikembalikan dengan `expm1`. Metrik di bawah: RMSE (ganti bila metrik lomba lain).

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/stacking_teks_tabular_regresi"
OOF_FOLDER = "../../../outputs/text/oof"

DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"   # GANTI saat lomba: sample dari panitia
KEY_CSV = f"{DATA_FOLDER}/baju_wanita_test_labels.csv"       # GANTI saat lomba (kunci, kalau ada)

ID_COL = "id"                                                # GANTI saat lomba
TARGET = "rating"                                            # GANTI saat lomba
SUB_COL = "rating"                                           # GANTI saat lomba: kolom sample
TEXT_COLS = ["title", "review_text"]                         # GANTI saat lomba
CAT_COLS = ["clothing_id", "division_name", "department_name", "class_name"]   # GANTI saat lomba
DROP_COLS = ["recommended_ind"]                              # GANTI saat lomba: hasil cek bocor
LOG_TARGET = False                                           # GANTI: True untuk harga/jumlah

MODEL_NAME = "lgbm_stacking_regresi"
N_FOLDS = 5
SEED = 42
N_BINS = 5                  # bin kuantil target untuk fitur kelas-bin
N_NEIGHBORS = 20
EXTRA_OOF = []              # OOF regresi lain di OOF_FOLDER (n × 1), mis. ["ft_regresi"]


def rmse(truth, pred):
    return np.sqrt(mean_squared_error(truth, pred))       # GANTI: metrik lomba (kecil = baik)


print("uji rmse:", rmse(np.array([1.0, 2.0]), np.array([1.0, 4.0])))

## Baca data

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
for col in TEXT_COLS:
    train[col] = train[col].fillna("").astype(str)
    test[col] = test[col].fillna("").astype(str)
for col in CAT_COLS:
    train[col] = train[col].fillna("kosong").astype(str)
    test[col] = test[col].fillna("kosong").astype(str)
full_train = train[TEXT_COLS].agg(" . ".join, axis=1)
full_test = test[TEXT_COLS].agg(" . ".join, axis=1)

skip = [ID_COL, TARGET] + TEXT_COLS + CAT_COLS + DROP_COLS
NUM_COLS = []
for col in train.columns:
    if col not in skip:
        NUM_COLS.append(col)
target = train[TARGET].astype(float).to_numpy()
y = target
if LOG_TARGET:
    y = np.log1p(target)
print("train:", train.shape, "| test:", test.shape, "| kolom angka:", NUM_COLS)
summary = pd.Series(target).describe()
print(summary.round(3))

## Buat sample_submission demo (HAPUS saat lomba)

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUB_COL] = float(np.median(target))
demo_sample.to_csv(SAMPLE_CSV, index=False)
print("sample demo ditulis:", SAMPLE_CSV, demo_sample.shape)

## Lipatan + TF-IDF + bin target
Bin = kuantil target (dihitung di seluruh train; hanya dipakai sebagai label model teks di
dalam lipatan, jadi tidak bocor ke baris validasi). Target dengan sedikit nilai unik (rating
1–5) menghasilkan bin < `N_BINS` (duplikat dibuang).

In [ ]:
folds = list(KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(full_train))
all_text = pd.concat([full_train, full_test], ignore_index=True)
word_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=200000, sublinear_tf=True)
char_tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3,
                             max_features=200000, sublinear_tf=True)
matrix = hstack([word_tfidf.fit_transform(all_text), char_tfidf.fit_transform(all_text)])
matrix = matrix.tocsr()
matrix_train = matrix[: len(train)]
matrix_test = matrix[len(train):]
bins = pd.qcut(y, N_BINS, labels=False, duplicates="drop")
N_BIN_CLASSES = int(bins.max()) + 1
print("TF-IDF:", matrix_train.shape, "| bin:", np.bincount(bins))

## Fitur 1: Ridge OOF + kelas-bin OOF (+ `EXTRA_OOF`)

In [ ]:
def oof_ridge(model, X, X_test):
    oof = np.zeros(X.shape[0], dtype="float32")
    test_pred = np.zeros(X_test.shape[0], dtype="float32")
    for idx_fit, idx_val in folds:
        fitted = clone(model).fit(X[idx_fit], y[idx_fit])
        oof[idx_val] = fitted.predict(X[idx_val])
        test_pred += fitted.predict(X_test) / N_FOLDS
    return oof, test_pred


def oof_bins(model, X, X_test):
    oof = np.zeros((X.shape[0], N_BIN_CLASSES), dtype="float32")
    test_pred = np.zeros((X_test.shape[0], N_BIN_CLASSES), dtype="float32")
    for idx_fit, idx_val in folds:
        fitted = clone(model).fit(X[idx_fit], bins[idx_fit])
        oof[idx_val] = fitted.predict_proba(X[idx_val])
        test_pred += fitted.predict_proba(X_test) / N_FOLDS
    return oof, test_pred


START = time.time()
ridge_oof, ridge_test = oof_ridge(Ridge(alpha=2.0), matrix_train, matrix_test)
print("ridge OOF RMSE:", round(rmse(y, ridge_oof), 4), "| detik", round(time.time() - START))
bin_oof, bin_test = oof_bins(LogisticRegression(C=4.0, max_iter=2000), matrix_train, matrix_test)
print("kelas-bin selesai | detik", round(time.time() - START))

Rakit fitur OOF teks. Nilai harapan dari peluang bin (`Σ peluang × rata-rata target bin`) ikut
jadi satu kolom: tebakan angka kedua dari model klasifikasi.

In [ ]:
bin_means = np.bincount(bins, weights=y) / np.bincount(bins)      # rata-rata target tiap bin
level1_train = [ridge_oof[:, None], bin_oof, (bin_oof @ bin_means)[:, None]]
level1_test = [ridge_test[:, None], bin_test, (bin_test @ bin_means)[:, None]]
level1_names = ["ridge"]
for number in range(N_BIN_CLASSES):
    level1_names.append(f"bin_p{number}")
level1_names.append("bin_harapan")
print("bin_harapan OOF RMSE:", round(rmse(y, bin_oof @ bin_means), 4))
for name in EXTRA_OOF:
    extra = np.load(f"{OOF_FOLDER}/oof_{name}.npy").reshape(len(train), -1)
    print(f"{name}: {extra.shape} | RMSE {rmse(y, extra[:, 0]):.4f}")
    level1_train.append(extra)
    level1_test.append(np.load(f"{OOF_FOLDER}/test_{name}.npy").reshape(len(test), -1))
    level1_names.append(name)

## Fitur 2: tetangga terdekat (target tetangga)
Per baris: rata-rata target tetangga berbobot kemiripan, simpangan bakunya, target tetangga
ke-1, kemiripan ke-1 dan rata-rata. Dihitung per lipatan; test = rata-rata lima lipatan.

In [ ]:
def neighbor_block(ref, ref_y, query):
    finder = NearestNeighbors(n_neighbors=N_NEIGHBORS, metric="cosine").fit(ref)
    distance, index = finder.kneighbors(query)
    similarity = np.clip(1 - distance, 1e-6, None)
    values = ref_y[index]
    mean = (similarity * values).sum(axis=1) / similarity.sum(axis=1)
    spread = np.sqrt((similarity * (values - mean[:, None]) ** 2).sum(axis=1)
                     / similarity.sum(axis=1))
    return np.column_stack([mean, spread, values[:, 0], similarity[:, 0], similarity.mean(axis=1)])


neighbor_names = ["tetangga_rata", "tetangga_sebaran", "tetangga_1", "tetangga_sim1",
                  "tetangga_sim_rata"]
neighbor_train = np.zeros((len(train), len(neighbor_names)), dtype="float32")
neighbor_test = np.zeros((len(test), len(neighbor_names)), dtype="float32")
START = time.time()
for idx_fit, idx_val in folds:
    ref = matrix_train[idx_fit]
    neighbor_train[idx_val] = neighbor_block(ref, y[idx_fit], matrix_train[idx_val])
    neighbor_test += neighbor_block(ref, y[idx_fit], matrix_test) / N_FOLDS
print("kNN saja RMSE:", round(rmse(y, neighbor_train[:, 0]), 4), "| detik",
      round(time.time() - START))

## Fitur 3: statistik teks + rakit fitur

In [ ]:
def text_stats(table, col):
    text = table[col].fillna("")
    out = pd.DataFrame(index=table.index)
    out[f"{col}_n_char"] = text.str.len()
    out[f"{col}_n_word"] = text.str.split().str.len()
    out[f"{col}_n_digit"] = text.str.count(r"\d")
    out[f"{col}_n_excl"] = text.str.count("!")
    out[f"{col}_upper_ratio"] = text.str.count(r"[A-Z]") / (out[f"{col}_n_char"] + 1)
    return out


all_data = pd.concat([train, test], ignore_index=True)
stat_blocks = []
for col in TEXT_COLS:
    stat_blocks.append(text_stats(all_data, col))
stats = pd.concat(stat_blocks, axis=1)
level1_all = np.vstack([np.hstack(level1_train), np.hstack(level1_test)])
blocks = [all_data[NUM_COLS], stats, pd.DataFrame(level1_all, columns=level1_names),
          pd.DataFrame(np.vstack([neighbor_train, neighbor_test]), columns=neighbor_names)]
for col in CAT_COLS:
    blocks.append(all_data[col].astype("category"))
features = pd.concat(blocks, axis=1)
X_train = features.iloc[: len(train)].reset_index(drop=True)
X_test = features.iloc[len(train):].reset_index(drop=True)
GROUPS = {"tabel": list(NUM_COLS) + list(CAT_COLS), "statistik": list(stats.columns),
          "oof_teks": level1_names, "tetangga": neighbor_names}
print("fitur:", X_train.shape)
for group in GROUPS:
    print(" ", group, len(GROUPS[group]))

## Model + CV (lipatan yang sama)
Alternatif ber-`#`: objektif Huber (pencilan) atau MAE (metrik MAE).

In [ ]:
model = LGBMRegressor(n_estimators=600, learning_rate=0.03, num_leaves=15, min_child_samples=40,
                      colsample_bytree=0.4, subsample=0.8, subsample_freq=1, reg_lambda=1.0,
                      random_state=SEED, n_jobs=-1, verbose=-1)
# model.set_params(objective="huber")   # alt: target berpencilan
# model.set_params(objective="mae")     # alt: metrik lomba MAE


def run_cv(model, X, X_test, verbose=True):
    oof = np.zeros(len(X), dtype="float32")
    test_pred = np.zeros(len(X_test), dtype="float32")
    for fold, (idx_fit, idx_val) in enumerate(folds):
        fitted = copy.deepcopy(model)
        fitted.fit(X.iloc[idx_fit], y[idx_fit])
        oof[idx_val] = fitted.predict(X.iloc[idx_val])
        test_pred += fitted.predict(X_test) / N_FOLDS
        if verbose:
            print("lipatan", fold, "RMSE", round(rmse(y[idx_val], oof[idx_val]), 4))
    return oof, test_pred


START = time.time()
oof, test_pred = run_cv(model, X_train, X_test)
SECONDS = round(time.time() - START)

## Evaluasi OOF (skala asli target)
Target bulat (rating): bulatkan dengan ambang yang dioptimalkan (`arsenal/tabular/metrik` #6)
bila metrik lomba QWK/akurasi.

In [ ]:
pred_oof = oof
if LOG_TARGET:
    pred_oof = np.expm1(oof)
SCORE = rmse(target, pred_oof)
print("RMSE OOF:", round(SCORE, 4), "| MAE:", round(mean_absolute_error(target, pred_oof), 4),
      "| R2:", round(r2_score(target, pred_oof), 4), "| detik:", SECONDS)
baseline = np.full(len(target), target.mean())
print("pembanding: tebak rata-rata RMSE", round(rmse(target, baseline), 4))
figure, ax = plt.subplots(figsize=(6, 5))
ax.scatter(target, pred_oof, s=4, alpha=0.3, color="#0072B2")
ax.plot([target.min(), target.max()], [target.min(), target.max()], color="#1a1a1a", linewidth=1)
ax.set_xlabel("target asli")
ax.set_ylabel("tebakan OOF")
figure.savefig(f"{OUTPUT_FOLDER}/tebakan_vs_asli.png", dpi=200, bbox_inches="tight")
plt.show()

## Ablasi + pentingnya kelompok fitur (bahan slide)

In [ ]:
ablation_model = copy.deepcopy(model).set_params(n_estimators=300, learning_rate=0.06)
columns = []
rows = []
for group in ["tabel", "statistik", "oof_teks", "tetangga"]:
    columns = columns + GROUPS[group]
    if len(columns) == 0:
        continue
    ablation_oof, _ = run_cv(ablation_model, X_train[columns], X_test[columns], False)
    rows.append({"fitur sampai": group, "kolom": len(columns),
                 "RMSE OOF (skala model)": round(rmse(y, ablation_oof), 4)})
ablation = pd.DataFrame(rows)
ablation.to_csv(f"{OUTPUT_FOLDER}/ablasi.csv", index=False)
print(ablation.to_string(index=False))
full_model = copy.deepcopy(model).fit(X_train, y)
importance = pd.Series(full_model.feature_importances_, index=X_train.columns)
by_group = {}
for group in GROUPS:
    by_group[group] = importance.reindex(GROUPS[group]).sum() / importance.sum()
by_group = pd.Series(by_group).sort_values()
by_group.rename("porsi").to_csv(f"{OUTPUT_FOLDER}/kelompok_fitur.csv")
print(by_group.round(3).to_dict())

## Simpan OOF (n × 1, skala model) + catat percobaan

In [ ]:
np.save(f"{OOF_FOLDER}/oof_{MODEL_NAME}.npy", oof[:, None].astype("float32"))
np.save(f"{OOF_FOLDER}/test_{MODEL_NAME}.npy", test_pred[:, None].astype("float32"))
log_path = f"{OOF_FOLDER}/percobaan.csv"
log_row = pd.DataFrame([{"waktu": time.strftime("%Y-%m-%d %H:%M:%S"),
                         "notebook": "stacking_teks_tabular_regresi", "model": MODEL_NAME,
                         "metrik": "rmse", "skor_cv": round(SCORE, 4), "detik": SECONDS,
                         "catatan": f"{X_train.shape[1]} fitur, log {LOG_TARGET}"}])
log_row.to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
print("disimpan:", f"oof_{MODEL_NAME}.npy")

## Submission + cek

In [ ]:
pred_test = test_pred
if LOG_TARGET:
    pred_test = np.expm1(test_pred)
sample = pd.read_csv(SAMPLE_CSV)
pred_by_id = pd.Series(pred_test, index=test[ID_COL].values)
submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUB_COL] = sample[ID_COL].map(pred_by_id).values
submission.to_csv(f"{OUTPUT_FOLDER}/submission.csv", index=False)
written = pd.read_csv(f"{OUTPUT_FOLDER}/submission.csv")
problems = []
if list(written.columns) != list(sample.columns):
    problems.append(f"kolom/urutan beda: {list(written.columns)} vs {list(sample.columns)}")
if len(written) != len(sample) or written[SUB_COL].isna().sum() > 0:
    problems.append("jumlah baris beda atau ada tebakan kosong")
print(written[SUB_COL].describe().round(3))
print("MASALAH:", len(problems), problems)

## Cek dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].values, index=key[ID_COL].values)
truth = submission[ID_COL].map(key_by_id).astype(float)
print("RMSE test:", round(rmse(truth, submission[SUB_COL]), 4), "| RMSE OOF:", round(SCORE, 4))